In [ ]:
# Imports and Configuration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')

# Configuration
RESULTS_PATH = '../results/results.csv'
FIGURE_DIR = '../figures'

# Display settings
pd.set_option('display.precision', 4)
%matplotlib inline

In [ ]:
# Load Results
df = pd.read_csv(RESULTS_PATH)

# Verify data integrity
print("Results loaded successfully!")
print(f"Total experiments: {len(df)}")
print(f"\nMethods: {df['method'].unique()}")
print(f"Components: {sorted(df['components'].unique())}")
print(f"Seeds: {sorted(df['seed'].unique())}")

# Check for any errors
if df['accuracy'].dtype == 'object':
    errors = df[df['accuracy'] == 'ERROR']
    if len(errors) > 0:
        print(f"\nWARNING: {len(errors)} failed experiments")
        df = df[df['accuracy'] != 'ERROR']
        df['accuracy'] = df['accuracy'].astype(float)

df.head(10)

In [ ]:
# Aggregate Statistics
# Compute mean ± std for each method and component count

summary = df.groupby(['method', 'components'])['accuracy'].agg(['mean', 'std', 'count'])
summary.columns = ['mean_accuracy', 'std_accuracy', 'n_runs']
summary = summary.reset_index()

# Pivot for easier viewing
pivot_mean = summary.pivot(index='components', columns='method', values='mean_accuracy')
pivot_std = summary.pivot(index='components', columns='method', values='std_accuracy')

print("Mean Accuracy by Method and Components:")
print("="*50)
display(pivot_mean.round(4))

print("\nStandard Deviation:")
print("="*50)
display(pivot_std.round(4))

In [ ]:
# Main Plot: Accuracy vs Components
# One curve per method with error bands

fig, ax = plt.subplots(figsize=(10, 6))

colors = {'lda': '#1f77b4', 'pca': '#ff7f0e', 'rp': '#2ca02c'}
labels = {'lda': 'LDA (Supervised)', 'pca': 'PCA (Unsupervised)', 'rp': 'Random Projection'}

for method in ['lda', 'pca', 'rp']:
    method_data = summary[summary['method'] == method].sort_values('components')
    
    x = method_data['components']
    y = method_data['mean_accuracy']
    yerr = method_data['std_accuracy']
    
    ax.plot(x, y, 'o-', color=colors[method], label=labels[method], linewidth=2, markersize=8)
    ax.fill_between(x, y - yerr, y + yerr, color=colors[method], alpha=0.2)

ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('CIFAR-100 Classification Accuracy vs Dimensionality', fontsize=14)
ax.legend(loc='lower right', fontsize=11)
ax.set_xticks([2, 5, 10, 20, 40, 80, 99])
ax.set_xlim(0, 105)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/accuracy_vs_components.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Relative Improvement Plot
# Shows how much each additional component helps

fig, ax = plt.subplots(figsize=(10, 6))

for method in ['lda', 'pca', 'rp']:
    method_data = summary[summary['method'] == method].sort_values('components')
    
    x = method_data['components'].values
    y = method_data['mean_accuracy'].values
    
    # Compute marginal improvement (accuracy gain per component added)
    marginal_improvement = np.diff(y) / np.diff(x)
    x_mid = (x[:-1] + x[1:]) / 2
    
    ax.plot(x_mid, marginal_improvement * 100, 'o-', color=colors[method], 
            label=labels[method], linewidth=2, markersize=8)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Number of Components (midpoint)', fontsize=12)
ax.set_ylabel('Marginal Accuracy Gain (% per component)', fontsize=12)
ax.set_title('Diminishing Returns Analysis', fontsize=14)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/marginal_improvement.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# LDA Advantage Plot
# Shows LDA's improvement over PCA at each component count

fig, ax = plt.subplots(figsize=(10, 6))

lda_data = summary[summary['method'] == 'lda'].set_index('components')['mean_accuracy']
pca_data = summary[summary['method'] == 'pca'].set_index('components')['mean_accuracy']
rp_data = summary[summary['method'] == 'rp'].set_index('components')['mean_accuracy']

components = sorted(df['components'].unique())

lda_vs_pca = [(lda_data[c] - pca_data[c]) * 100 for c in components]
lda_vs_rp = [(lda_data[c] - rp_data[c]) * 100 for c in components]

x = np.arange(len(components))
width = 0.35

ax.bar(x - width/2, lda_vs_pca, width, label='LDA - PCA', color='#1f77b4')
ax.bar(x + width/2, lda_vs_rp, width, label='LDA - RP', color='#2ca02c')

ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5)
ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('LDA Advantage (% points)', fontsize=12)
ax.set_title('LDA Performance Advantage Over Baselines', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(components)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../figures/lda_advantage.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary Table for Paper
# Clean table with mean ± std format

def format_mean_std(row):
    return f"{row['mean_accuracy']:.4f} ± {row['std_accuracy']:.4f}"

summary['formatted'] = summary.apply(format_mean_std, axis=1)
paper_table = summary.pivot(index='components', columns='method', values='formatted')
paper_table = paper_table[['lda', 'pca', 'rp']]  # Reorder columns
paper_table.columns = ['LDA', 'PCA', 'Random Projection']

print("Table for Paper:")
print("="*70)
display(paper_table)

## Runtime Analysis

The following cells analyze timing data captured during experiments.
This helps understand practical trade-offs between accuracy and computational cost.

In [ ]:
# Runtime Summary Statistics
# Aggregate timing data by method and components

timing_cols = ['reduction_fit_sec', 'reduction_transform_train_sec', 
               'reduction_transform_test_sec', 'classifier_train_sec', 
               'evaluation_sec', 'total_runtime_sec']

# Check if timing columns exist
if 'total_runtime_sec' in df.columns:
    timing_summary = df.groupby(['method', 'components'])[timing_cols].agg(['mean', 'std'])
    
    print("Mean Runtime (seconds) by Method and Components:")
    print("="*70)
    
    # Show total runtime pivot
    pivot_runtime = df.groupby(['method', 'components'])['total_runtime_sec'].mean().reset_index()
    pivot_runtime = pivot_runtime.pivot(index='components', columns='method', values='total_runtime_sec')
    display(pivot_runtime.round(3))
else:
    print("No timing data found. Run experiments with the updated script to capture timing.")

In [ ]:
# Runtime vs Components Plot
# Shows how runtime scales with dimensionality for each method

if 'total_runtime_sec' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: Total runtime
    ax1 = axes[0]
    runtime_agg = df.groupby(['method', 'components'])['total_runtime_sec'].agg(['mean', 'std']).reset_index()
    
    for method in ['lda', 'pca', 'rp']:
        method_data = runtime_agg[runtime_agg['method'] == method].sort_values('components')
        x = method_data['components']
        y = method_data['mean']
        yerr = method_data['std']
        
        ax1.plot(x, y, 'o-', color=colors[method], label=labels[method], linewidth=2, markersize=8)
        ax1.fill_between(x, y - yerr, y + yerr, color=colors[method], alpha=0.2)
    
    ax1.set_xlabel('Number of Components', fontsize=12)
    ax1.set_ylabel('Total Runtime (seconds)', fontsize=12)
    ax1.set_title('Total Runtime vs Dimensionality', fontsize=14)
    ax1.legend(loc='upper left', fontsize=10)
    ax1.set_xticks([2, 5, 10, 20, 40, 80, 99])
    ax1.grid(True, alpha=0.3)
    
    # Right: Reduction fit time only (where methods differ most)
    ax2 = axes[1]
    fit_agg = df.groupby(['method', 'components'])['reduction_fit_sec'].agg(['mean', 'std']).reset_index()
    
    for method in ['lda', 'pca', 'rp']:
        method_data = fit_agg[fit_agg['method'] == method].sort_values('components')
        x = method_data['components']
        y = method_data['mean']
        yerr = method_data['std']
        
        ax2.plot(x, y, 'o-', color=colors[method], label=labels[method], linewidth=2, markersize=8)
        ax2.fill_between(x, y - yerr, y + yerr, color=colors[method], alpha=0.2)
    
    ax2.set_xlabel('Number of Components', fontsize=12)
    ax2.set_ylabel('Reduction Fit Time (seconds)', fontsize=12)
    ax2.set_title('Dimensionality Reduction Fitting Time', fontsize=14)
    ax2.legend(loc='upper left', fontsize=10)
    ax2.set_xticks([2, 5, 10, 20, 40, 80, 99])
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../figures/runtime_vs_components.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No timing data available.")

In [ ]:
# Accuracy vs Runtime Trade-off (Pareto Analysis)
# Shows which methods provide best accuracy per unit of computation

if 'total_runtime_sec' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for method in ['lda', 'pca', 'rp']:
        method_df = df[df['method'] == method]
        method_agg = method_df.groupby('components').agg({
            'accuracy': 'mean',
            'total_runtime_sec': 'mean'
        }).reset_index()
        
        ax.scatter(method_agg['total_runtime_sec'], method_agg['accuracy'], 
                   color=colors[method], s=100, label=labels[method], alpha=0.8)
        
        # Connect points with line
        method_agg_sorted = method_agg.sort_values('components')
        ax.plot(method_agg_sorted['total_runtime_sec'], method_agg_sorted['accuracy'],
                color=colors[method], linestyle='--', alpha=0.5)
        
        # Annotate with component count
        for _, row in method_agg.iterrows():
            ax.annotate(f"{int(row['components'])}", 
                       (row['total_runtime_sec'], row['accuracy']),
                       textcoords="offset points", xytext=(5, 5), fontsize=8)
    
    ax.set_xlabel('Mean Runtime (seconds)', fontsize=12)
    ax.set_ylabel('Mean Accuracy', fontsize=12)
    ax.set_title('Accuracy vs Runtime Trade-off (numbers = components)', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../figures/accuracy_runtime_tradeoff.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No timing data available.")

In [ ]:
# Runtime Breakdown Stacked Bar Chart
# Shows where time is spent for each method at 99 components

if 'total_runtime_sec' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Get data for 99 components (maximum)
    df_99 = df[df['components'] == 99]
    
    timing_components = ['reduction_fit_sec', 'reduction_transform_train_sec', 
                         'reduction_transform_test_sec', 'classifier_train_sec', 'evaluation_sec']
    timing_labels = ['Reduction Fit', 'Transform Train', 'Transform Test', 
                     'Classifier Train', 'Evaluation']
    
    breakdown = df_99.groupby('method')[timing_components].mean()
    breakdown = breakdown.reindex(['lda', 'pca', 'rp'])
    
    # Create stacked bar
    bottom = np.zeros(3)
    bar_colors = plt.cm.Set3(np.linspace(0, 1, len(timing_components)))
    
    x_pos = np.arange(3)
    for i, (col, label) in enumerate(zip(timing_components, timing_labels)):
        values = breakdown[col].values
        ax.bar(x_pos, values, bottom=bottom, label=label, color=bar_colors[i], width=0.6)
        bottom += values
    
    ax.set_xlabel('Method', fontsize=12)
    ax.set_ylabel('Time (seconds)', fontsize=12)
    ax.set_title('Runtime Breakdown at 99 Components', fontsize=14)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['LDA', 'PCA', 'Random Projection'])
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('../figures/runtime_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No timing data available.")

In [ ]:
# Feature Extraction Timing (if available)
import json
import os

extraction_timing_path = '../features/saved/extraction_timing.json'

if os.path.exists(extraction_timing_path):
    with open(extraction_timing_path, 'r') as f:
        extraction_timing = json.load(f)
    
    print("FEATURE EXTRACTION TIMING")
    print("="*50)
    for key, value in extraction_timing.items():
        print(f"  {key}: {value:.2f}s")
else:
    print("Feature extraction timing not found.")
    print("Run feature extraction with the updated script to capture timing.")

In [ ]:
# Complete Timing Table for Paper
# Includes both accuracy and runtime

if 'total_runtime_sec' in df.columns:
    full_summary = df.groupby(['method', 'components']).agg({
        'accuracy': ['mean', 'std'],
        'total_runtime_sec': ['mean', 'std']
    }).reset_index()
    
    full_summary.columns = ['method', 'components', 'acc_mean', 'acc_std', 'time_mean', 'time_std']
    
    def format_acc_time(row):
        acc_str = f"{row['acc_mean']:.4f} ± {row['acc_std']:.4f}"
        time_str = f"{row['time_mean']:.3f}s"
        return acc_str, time_str
    
    print("Complete Results Table (Accuracy | Runtime):")
    print("="*80)
    
    for method in ['lda', 'pca', 'rp']:
        print(f"\n{labels[method]}:")
        method_data = full_summary[full_summary['method'] == method].sort_values('components')
        for _, row in method_data.iterrows():
            print(f"  {int(row['components']):3d} components: "
                  f"Acc = {row['acc_mean']:.4f} ± {row['acc_std']:.4f}, "
                  f"Time = {row['time_mean']:.3f}s ± {row['time_std']:.3f}s")
else:
    print("No timing data available.")

In [ ]:
# Key Statistics Summary

print("KEY FINDINGS")
print("="*70)

# Best accuracy for each method
for method in ['lda', 'pca', 'rp']:
    method_summary = summary[summary['method'] == method]
    best = method_summary.loc[method_summary['mean_accuracy'].idxmax()]
    print(f"\n{labels[method]}:")
    print(f"  Best accuracy: {best['mean_accuracy']:.4f} ± {best['std_accuracy']:.4f}")
    print(f"  At {int(best['components'])} components")

# Find saturation point (where improvement < 0.5% per component added)
print("\n" + "-"*70)
print("Saturation Analysis (improvement < 0.5% per component added):")

for method in ['lda', 'pca', 'rp']:
    method_data = summary[summary['method'] == method].sort_values('components')
    x = method_data['components'].values
    y = method_data['mean_accuracy'].values
    
    marginal = np.diff(y) / np.diff(x) * 100
    
    for i, (c1, c2, m) in enumerate(zip(x[:-1], x[1:], marginal)):
        if m < 0.5:
            print(f"  {labels[method]}: saturates around {int(c1)} components")
            break
    else:
        print(f"  {labels[method]}: no clear saturation point")

## Observations

**Note:** This section documents observed trends only. Conclusions and interpretations are left to the human researcher.

### Observed Trends:

1. **Accuracy vs Components**: [To be filled after running experiments]
   - All methods show increasing accuracy with more components
   - The rate of improvement decreases at higher component counts

2. **LDA vs PCA**: [To be filled after running experiments]
   - Comparison of supervised vs unsupervised reduction

3. **Saturation Behavior**: [To be filled after running experiments]
   - Where each method reaches diminishing returns

4. **Variance Across Seeds**: [To be filled after running experiments]
   - Stability of results across random seeds

---

*Analysis generated from saved experimental results. No model training performed in this notebook.*